# Fisher-KPP RK4 Demo

This notebook runs both 1D and 2D Fisher-KPP RK4 examples.

- 1D: traveling front with Dirichlet boundaries.
- 2D: square-domain Gaussian seed with no-flux boundaries, matching the companion forward PINN/RK4 setup.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fisher_kpp_rk4 import check_rk4_stability, estimate_front_speed, solve_rk4, solve_rk4_2d
from fisher_kpp_rk4.config import (
    BOX_2D,
    D,
    D_2D,
    GRID_2D,
    L,
    Nt,
    Nt_2d,
    Nx,
    T,
    T_2D,
    dt,
    dt_2d,
    dx,
    dx_2d,
    initial_condition,
    initial_condition_2d,
    left_bc,
    r,
    r_2D,
    right_bc,
    x,
    x_2d,
    y_2d,
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


## 1D Setup and Solve

In [ ]:
info_1d = check_rk4_stability(dx=dx, dt=dt, D=D, r=r, dim=1)
print(f"1D: D={D}, r={r}, L={L}, T={T}")
print(f"Nx={Nx}, dx={dx:.6g}, Nt={Nt}, dt={dt:.6g}")
print(f"Practical dt safe? {info_1d['is_practically_safe']} (limit={info_1d['dt_practical']:.6g})")

result_1d = solve_rk4(
    x=x,
    dt=dt,
    Nt=Nt,
    D=D,
    r=r,
    initial_condition=initial_condition,
    left_bc=left_bc,
    right_bc=right_bc,
    save_interval=5.0,
)

c_min = 2.0 * np.sqrt(D * r)
c_num = estimate_front_speed(result_1d["times"], result_1d["fronts"], t_min=5.0, x_max=0.85 * L)
print(f"Theoretical minimal KPP speed c* = {c_min:.6g}")
print(f"Estimated front speed before boundary interaction = {c_num:.6g}")


## 1D Visualizations

In [ ]:
plt.figure(figsize=(8, 5))
for t, u in zip(result_1d["times"], result_1d["snapshots"]):
    plt.plot(result_1d["x"], u, label=f"t={t:.0f}")
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.ylim(-0.05, 1.05)
plt.title("1D Fisher-KPP solved by MOL-FDM + RK4")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(result_1d["times"], result_1d["fronts"], marker="o")
plt.xlabel("t")
plt.ylabel("front position, u=0.5")
plt.title("1D front propagation")
plt.tight_layout()
plt.show()


## 2D Setup and Solve

In [ ]:
info_2d = check_rk4_stability(dx=dx_2d, dt=dt_2d, D=D_2D, r=r_2D, dim=2)
print(f"2D: D={D_2D}, r={r_2D}, box={BOX_2D}, T={T_2D}")
print(f"grid={GRID_2D}x{GRID_2D}, dx={dx_2d:.6g}, Nt={Nt_2d}, dt={dt_2d:.6g}")
print(f"Practical dt safe? {info_2d['is_practically_safe']} (limit={info_2d['dt_practical']:.6g})")

result_2d = solve_rk4_2d(
    x=x_2d,
    y=y_2d,
    dt=dt_2d,
    Nt=Nt_2d,
    D=D_2D,
    r=r_2D,
    initial_condition=initial_condition_2d,
    save_interval=0.05,
)
print(f"Final mean mass = {result_2d['mass'][-1]:.6g}")
print(f"Final area u>=0.05 = {result_2d['area_ge_0.05'][-1]:.6g}")
print(f"Final area u>=0.10 = {result_2d['area_ge_0.10'][-1]:.6g}")


## 2D Visualizations

In [ ]:
snapshots = result_2d["snapshots"]
times = result_2d["times"]
panel_idx = np.unique(np.linspace(0, len(times) - 1, 4, dtype=int))
fig, axes = plt.subplots(1, len(panel_idx), figsize=(4.0 * len(panel_idx), 3.4), constrained_layout=True)
if len(panel_idx) == 1:
    axes = [axes]
for ax, idx in zip(axes, panel_idx):
    im = ax.imshow(
        snapshots[idx].T,
        origin="lower",
        extent=[result_2d["x"][0], result_2d["x"][-1], result_2d["y"][0], result_2d["y"][-1]],
        vmin=0.0,
        vmax=1.0,
        cmap="magma",
    )
    ax.set_title(f"t={times[idx]:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
fig.colorbar(im, ax=axes, shrink=0.82)
fig.suptitle("2D Fisher-KPP RK4 snapshots")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(result_2d["times"], result_2d["mass"], label="mean mass")
plt.plot(result_2d["times"], result_2d["area_ge_0.05"], label="area u>=0.05")
plt.plot(result_2d["times"], result_2d["area_ge_0.10"], label="area u>=0.10")
plt.xlabel("t")
plt.ylabel("fraction")
plt.title("2D mass and front-area diagnostics")
plt.legend()
plt.tight_layout()
plt.show()


## Save Outputs

In [ ]:
np.savez(OUTPUT_DIR / "fisher_kpp_rk4_1d_results.npz", **result_1d, D=D, r=r, L=L, T=T, dx=dx, dt=dt)
np.savez(OUTPUT_DIR / "fisher_kpp_rk4_2d_results.npz", **result_2d, D=D_2D, r=r_2D, box=BOX_2D, T=T_2D, dx=dx_2d, dt=dt_2d)
print("Saved 1D and 2D NPZ outputs under outputs/.")
